In [ ]:
# ============================================================================
# 02 — BUILD THE BENCHMARK
# ----------------------------------------------------------------------------
# For each eligible protein we pick TWO partners:
#   - one where the protein does its USUAL thing   (habit agrees with truth)
#   - one where it does the OPPOSITE               (habit misleads)
# The two are matched on how well-documented they are, so any accuracy gap
# cannot be blamed on one being more obscure than the other.
# Output: data/processed/sighabit_v1.jsonl
# ============================================================================
import json, hashlib, datetime
from pathlib import Path
import pandas as pd, numpy as np

ROOT = Path.cwd().parent
RAW, PROC = ROOT/"data"/"raw", ROOT/"data"/"processed"
PROC.mkdir(parents=True, exist_ok=True)

SEED       = 20260817
VERSION    = "v1"
MIN_REFS   = 2       # each edge needs >=2 supporting papers
MIN_SIGN   = 2       # protein needs >=2 targets of each sign
MIN_USUAL  = 0.70    # protein's default must be at least 70% one-directional
REF_TOL    = 0.60    # matched pair: log-reference counts within this distance
N_PER_GROUP = 100    # pairs per protein group

rng = np.random.RandomState(SEED)

# ---- 1. rebuild the edge table (same filters as notebook 01) -------------
def is_complex(s):
    return (not isinstance(s,str)) or (not s.strip()) or "COMPLEX" in s.upper() or "_" in s

NEEDS_DIR = {"causal_ppi": True, "kinaseextra": True, "collectri": False}
raw, parts = {}, []
for name in ["causal_ppi","collectri","kinaseextra"]:
    raw[name] = pd.read_parquet(RAW/f"{name}.parquet")
    d = raw[name].copy()
    if NEEDS_DIR[name]:
        d = d[d.is_directed.astype(bool)]
    stim, inh = d.is_stimulation.astype(bool), d.is_inhibition.astype(bool)
    d = d[(stim ^ inh).values]
    d = d[~d.source_genesymbol.map(is_complex).values]
    d = d[~d.target_genesymbol.map(is_complex).values]
    d = d[~d.source.astype(str).str.upper().str.contains("COMPLEX").values]
    d = d[~d.target.astype(str).str.upper().str.contains("COMPLEX").values].copy()
    d["sign"]   = d.is_stimulation.astype(bool).map({True:"stimulates", False:"inhibits"})
    d["n_refs"] = d.n_references.fillna(0).astype(int) if "n_references" in d.columns else 0
    d["component"] = name
    parts.append(d[["source_genesymbol","target_genesymbol","source","target",
                    "sign","n_refs","component"]])

E = pd.concat(parts, ignore_index=True)
conf = E.groupby(["source_genesymbol","target_genesymbol"]).sign.nunique()
CONFLICTED = set(conf[conf>1].index)
E["_pk"] = list(zip(E.source_genesymbol, E.target_genesymbol))
E = E[~E._pk.isin(CONFLICTED)]
E = E.sort_values("n_refs", ascending=False).drop_duplicates("_pk", keep="first")
print(f"usable signed edges: {len(E):,}  (removed {len(CONFLICTED)} sign-conflicted pairs)")

# ---- 2. per-protein habit, LEAVE-ONE-OUT ---------------------------------
# The habit must never include the edge we are about to ask about, or the
# answer leaks into the very variable we use to predict it.
tot = E.groupby("source_genesymbol").sign.agg(
    n="size", n_stim=lambda s: (s=="stimulates").sum())

def loo_share(gene, sign_of_this_edge):
    """This protein's activation rate over its OTHER partners."""
    n, k = tot.loc[gene, "n"], tot.loc[gene, "n_stim"]
    n -= 1
    if sign_of_this_edge == "stimulates": k -= 1
    return k/n if n > 0 else np.nan

# ---- 3. eligible proteins -------------------------------------------------
G = pd.read_csv(PROC/"eligible_proteins.csv", index_col=0)
G["lopsided"] = np.maximum(G.stim_share, 1-G.stim_share)
G["usual"]    = np.where(G.stim_share > .5, "stimulates", "inhibits")
G = G[G.lopsided >= MIN_USUAL]

GROUP = {"kinase":"kinase", "transcription factor":"tf",
         "phosphatase":"other_enzyme", "E3 ligase":"other_enzyme",
         "protease":"other_enzyme", "other":"other_enzyme"}
G["group"] = G.pclass.map(GROUP)
print(f"\neligible proteins at >={MIN_USUAL:.0%} usual: {len(G)}")
print("  " + G.group.value_counts().to_string().replace("\n","\n  "))

# ---- 4. build matched pairs ------------------------------------------------
# For every protein: take one USUAL-sign partner and one OPPOSITE-sign partner
# whose reference counts are as close as possible. Reject if we cannot match.
Eq = E[E.n_refs >= MIN_REFS]
pairs, rejected = [], {"no_match":0, "too_few":0}

for gene in rng.permutation(G.index):
    r = G.loc[gene]
    usual, opp = r.usual, ("inhibits" if r.usual=="stimulates" else "stimulates")
    U = Eq[(Eq.source_genesymbol==gene) & (Eq.sign==usual)]
    O = Eq[(Eq.source_genesymbol==gene) & (Eq.sign==opp)]
    if len(U) < 1 or len(O) < 1:
        rejected["too_few"] += 1; continue
    # match on log reference count
    best = None
    for _, o in O.iterrows():
        d = (np.log1p(U.n_refs) - np.log1p(o.n_refs)).abs()
        j = d.idxmin()
        if d[j] <= REF_TOL and (best is None or d[j] < best[0]):
            best = (d[j], U.loc[j], o)
    if best is None:
        rejected["no_match"] += 1; continue
    _, u_edge, o_edge = best
    pairs.append(dict(gene=gene, group=r.group, pclass=r.pclass,
                      usual=usual, lopsided=r.lopsided,
                      u_target=u_edge.target_genesymbol, u_refs=int(u_edge.n_refs),
                      u_comp=u_edge.component, u_acc=u_edge.target,
                      o_target=o_edge.target_genesymbol, o_refs=int(o_edge.n_refs),
                      o_comp=o_edge.component, o_acc=o_edge.target,
                      src_acc=u_edge.source))
P = pd.DataFrame(pairs)
print(f"\nmatched pairs built: {len(P)}   rejected: {rejected}")

# ---- 5. quota by group, and balance the direction of the habit -----------
keep = []
for grp, g in P.groupby("group"):
    n = min(N_PER_GROUP, len(g))
    # within a group, keep habit direction as balanced as availability allows
    inh = g[g.usual=="inhibits"]; sti = g[g.usual=="stimulates"]
    n_inh = min(len(inh), n//2)
    n_sti = n - n_inh
    sel = pd.concat([inh.iloc[rng.permutation(len(inh))[:n_inh]],
                     sti.iloc[rng.permutation(len(sti))[:n_sti]]])
    keep.append(sel)
    print(f"  {grp:<14}{len(sel):>4} pairs  "
          f"(usual=inhibits {n_inh}, usual=stimulates {n_sti}, available {len(g)})")
P = pd.concat(keep).reset_index(drop=True)
print(f"\nfinal pairs: {len(P)}   ->  {len(P)*2} questions")

# ---- 6. expand each pair into two items -----------------------------------
def iid(*a): return hashlib.sha1("|".join(map(str,a)).encode()).hexdigest()[:16]

items = []
for _, p in P.iterrows():
    for cond, tgt, refs, comp, acc, truth in [
        ("habit_agrees",   p.u_target, p.u_refs, p.u_comp, p.u_acc, p.usual),
        ("habit_misleads", p.o_target, p.o_refs, p.o_comp, p.o_acc,
         "inhibits" if p.usual=="stimulates" else "stimulates")]:
        items.append(dict(
            item_id=iid(p.gene, tgt), pair_id=iid(p.gene, "pair"),
            source=p.gene, target=tgt,
            source_uniprot=p.src_acc, target_uniprot=acc,
            answer=truth, condition=cond,
            protein_usually=p.usual, group=p.group, pclass=p.pclass,
            lopsided=round(float(p.lopsided), 3),
            n_refs=int(refs), component=comp,
            src_stim_share_loo=round(float(loo_share(p.gene, truth)), 4),
            is_decoy=False))
B = pd.DataFrame(items)

# ---- 7. verify the design --------------------------------------------------
print("\n" + "="*78); print("DESIGN CHECK"); print("="*78)
print(f"items {len(B)} | proteins {B.source.nunique()} | pairs {B.pair_id.nunique()}")

ct = pd.crosstab(B.protein_usually, B.answer)
print("\nthe four cells (rows = protein's habit, cols = true answer here):")
print(ct.to_string())
print(f"\noverall answer balance: "
      f"stimulates {(B.answer=='stimulates').mean():.1%} / "
      f"inhibits {(B.answer=='inhibits').mean():.1%}")

print("\nby protein group:")
print(pd.crosstab(B.group, [B.protein_usually, B.condition]).to_string())

print("\nreference counts — are the two conditions matched?")
for c in ["habit_agrees","habit_misleads"]:
    s = B[B.condition==c]
    print(f"  {c:<16} median {s.n_refs.median():>5.0f}   "
          f"mean {s.n_refs.mean():>6.1f}   range {s.n_refs.min()}-{s.n_refs.max()}")
from scipy.stats import mannwhitneyu
u = mannwhitneyu(B[B.condition=='habit_agrees'].n_refs,
                 B[B.condition=='habit_misleads'].n_refs)
print(f"  Mann-Whitney p = {u.pvalue:.3f}  (want >0.05: conditions well matched)")

print("\nsanity checks:")
chk = [
 ("every protein contributes exactly 2 items",
  (B.groupby('source').size()==2).all()),
 ("no duplicate item ids", B.item_id.is_unique),
 ("answers balanced within 2pp",
  abs((B.answer=='stimulates').mean()-.5) < .02),
 ("each pair has one agree + one mislead",
  (B.groupby('pair_id').condition.nunique()==2).all()),
 ("no sign-conflicted pairs included",
  not any((r.source,r.target) in CONFLICTED for _,r in B.iterrows())),
]
for name, ok in chk:
    print(f"  [{'OK ' if ok else 'BAD'}] {name}")
assert all(o for _,o in chk), "design check failed"

# ---- 8. write ---------------------------------------------------------------
path = PROC/f"sighabit_{VERSION}.jsonl"
with open(path, "w", encoding="utf-8") as f:
    for _, r in B.iterrows():
        f.write(json.dumps({k:(None if pd.isna(v) else
                (int(v) if isinstance(v,(np.integer,)) else
                 float(v) if isinstance(v,(np.floating,)) else
                 bool(v) if isinstance(v,(np.bool_,)) else v))
                for k,v in r.items()}) + "\n")
SHA = hashlib.sha256(path.read_bytes()).hexdigest()[:16]

(PROC/f"sighabit_{VERSION}.meta.json").write_text(json.dumps(dict(
    version=VERSION, seed=SEED, built_utc=datetime.datetime.now(
        datetime.timezone.utc).isoformat(), sha256=SHA,
    n_items=len(B), n_proteins=int(B.source.nunique()),
    min_refs=MIN_REFS, min_targets_per_sign=MIN_SIGN,
    min_usual_share=MIN_USUAL, ref_match_tolerance=REF_TOL,
    n_sign_conflicted_removed=len(CONFLICTED),
    components=["causal_ppi","collectri","kinaseextra"]), indent=2))

print(f"\nwritten: {path}")
print(f"sha256 : {SHA}")
print("\nexample items:")
for _, r in B.head(4).iterrows():
    print(f"  {r.source:>9} -> {r.target:<10} truth={r.answer:<11} "
          f"({r.condition}, usually {r.protein_usually}, {r.n_refs} refs)")